## 1. O problema

O circuito e formado por uma fonte de tensao $E$, um resistor $R$ e um dispositivo nao-linear
cuja relacao tensao-corrente e $v = g(i)$.

Pela Lei das Tensoes de Kirchhoff:

$$E - R \cdot i - g(i) = 0$$

Com os valores do projeto ($E = 10\ \text{V}$, $R = 2\ \Omega$ e $g(i) = i^3$), o problema fica na forma $f(i) = 0$:

$$f(i) = 10 - 2i - i^3 = 0$$

A incognita $i$ e a corrente do circuito, em amperes.

In [31]:
from metodos_numericos import bissecao, newton_raphson, ponto_fixo
from problema import f, f_linha, phi, eps1, eps2, kmax

print("f(1) =", f(1))
print("f(2) =", f(2))

f(1) = 7
f(2) = -2


Como $f(1)$ e positivo e $f(2)$ e negativo, existe pelo menos uma raiz no intervalo $[1, 2]$.
Esse intervalo sera usado na bissecao, e o valor $i_0 = 1$ como chute inicial nos outros dois metodos.

## 2. Criterio de parada e precisao

Os tres metodos usam o mesmo criterio, para que a comparacao seja justa. Sao tres condicoes:

| Parametro | Valor | Para quando |
|---|---|---|
| `eps1` | $10^{-4}$ | $\lvert f(i) \rvert < eps1$ — o valor ja esta perto o bastante de zero |
| `eps2` | $10^{-4}$ | a variacao entre dois passos e menor que `eps2` — o metodo parou de avancar |
| `kmax` | 50 | limite de iteracoes, seguranca contra nao-convergencia |

**Justificativa da precisao.** A corrente procurada e da ordem de 1,8 A. Uma tolerancia de $10^{-4}$
garante quatro casas decimais, ou seja, erro abaixo de 0,1 mA — precisao muito acima da de qualquer
instrumento de bancada. Nao ha ganho pratico em exigir mais.

**Por que dois criterios.** Cada um cobre uma falha do outro. Perto de uma raiz onde a funcao e achatada,
$\lvert f(i) \rvert$ fica pequeno enquanto $i$ ainda muda bastante; ja em uma convergencia lenta, $i$ quase
para de variar enquanto $\lvert f(i) \rvert$ continua grande. Usando os dois ligados por *ou*, o metodo para
assim que qualquer uma das garantias for atingida.

**Por que `kmax`.** Nem todo metodo converge. O ponto fixo diverge se $\lvert \varphi'(i) \rvert > 1$, e o
Newton-Raphson pode oscilar com um chute ruim. Sem esse limite, o programa entraria em laco infinito.
O valor 50 e folgado: o metodo mais lento aqui precisou de 12 iteracoes.

## 3. Metodo da Bissecao

Divide o intervalo ao meio a cada passo e mantem a metade que contem a raiz, testando o sinal de
$f(a) \cdot f(i)$. Nao usa derivada e sempre converge, desde que $f(a)$ e $f(b)$ tenham sinais opostos.

In [27]:
raiz_bissecao, k_bissecao = bissecao(f, 1, 2, eps1, eps2, kmax)

print("\nraiz  =", raiz_bissecao)
print("|f(i)| =", abs(f(raiz_bissecao)))

iteracao 1 i = 1.5
iteracao 2 i = 1.75
iteracao 3 i = 1.875
iteracao 4 i = 1.8125
iteracao 5 i = 1.84375
iteracao 6 i = 1.859375
iteracao 7 i = 1.8515625
iteracao 8 i = 1.84765625
iteracao 9 i = 1.845703125
iteracao 10 i = 1.8466796875
iteracao 11 i = 1.84716796875
iteracao 12 i = 1.847412109375

raiz  = 1.847412109375
|f(i)| = 8.479623647872359e-05


## 4. Metodo de Newton-Raphson

Usa a reta tangente para estimar o proximo ponto: $i_{k+1} = i_k - f(i_k) / f'(i_k)$.
Precisa da derivada $f'(i) = -2 - 3i^2$, e em troca converge muito mais rapido (convergencia quadratica).

In [32]:
raiz_newton, k_newton = newton_raphson(f, f_linha, 1, eps1, eps2, kmax)

print("\nraiz  =", raiz_newton)
print("|f(i)| =", abs(f(raiz_newton)))

iteracao 1 i = 2.4
iteracao 2 i = 1.9526970954356846
iteracao 3 i = 1.852163495020807
iteracao 4 i = 1.8474292049246608
iteracao 5 i = 1.8474190378795425

raiz  = 1.8474190378795425
|f(i)| = 5.728990615239127e-10


## 5. Metodo do Ponto Fixo

Reescreve a equacao na forma $i = \varphi(i)$ e itera. Isolando $i$ em $10 - 2i - i^3 = 0$:

$$i^3 = 10 - 2i \quad \Longrightarrow \quad i = \sqrt[3]{10 - 2i} = \varphi(i)$$

Essa escolha de $\varphi$ converge porque $\lvert \varphi'(i) \rvert \approx 0{,}2$ perto da raiz, bem abaixo de 1.

In [29]:
raiz_ponto_fixo, k_ponto_fixo = ponto_fixo(phi, f, 1, eps1, eps2, kmax)

print("\nraiz  =", raiz_ponto_fixo)
print("|f(i)| =", abs(f(raiz_ponto_fixo)))

iteracao 1 i = 2.0
iteracao 2 i = 1.8171205928321397
iteracao 3 i = 1.8533184961168354
iteracao 4 i = 1.8462659533089356
iteracao 5 i = 1.847644247025129
iteracao 6 i = 1.8473750457659048
iteracao 7 i = 1.8474276309404878

raiz  = 1.8474276309404878
|f(i)| = 0.00010517034916546208


## 6. Resultados

In [30]:
resultados = [
    ("Bissecao", raiz_bissecao, k_bissecao),
    ("Newton-Raphson", raiz_newton, k_newton),
    ("Ponto Fixo", raiz_ponto_fixo, k_ponto_fixo),
]

print(f"{'Metodo':<16} {'Raiz (A)':>12} {'Iteracoes':>11} {'|f(i)|':>12}")
for nome, raiz, k in resultados:
    print(f"{nome:<16} {raiz:>12.6f} {k:>11} {abs(f(raiz)):>12.2e}")

Metodo               Raiz (A)   Iteracoes       |f(i)|
Bissecao             1.847412          12     8.48e-05
Newton-Raphson       1.847419           5     5.73e-10
Ponto Fixo           1.847428           7     1.05e-04


Os tres metodos chegam a mesma corrente, $i \approx 1{,}847$ A, dentro da tolerancia adotada.